# تحليل نتائج تجربة القياس — المرحلة 7
غلاف تفاعلي حول `evaluation/metrics.py` و`results/`:
1) الملخص الآلي `results/summary.json` (يولّده `analyze_results.py`)
2) مقاييس تُعاد محسوبتها هنا من الأزواج الخام (تحقق متقاطع)
3) رسوم Plotly تفاعلية + جدول الاتفاق مع التفسير

> ★ الحقيقة المرجعية تُملأ يدوياً أولاً؛ إن كانت فارغة سيظهر هنا تنبيه صريح بلا أرقام.

In [ ]:
import json, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / 'evaluation').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / 'results'
print('PROJECT_ROOT =', PROJECT_ROOT)

## 1) قراءة الملخص الآلي

In [ ]:
summary_path = RESULTS / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('أوضاع:', sorted(summary['modes']))
    print('GT جاهزة:', summary['ground_truth_ready'])
else:
    summary = None
    print('⚠️ لا يوجد results/summary.json — شغّل: python evaluation/analyze_results.py')

## 2) الأزواج الخام والتحقق المتقاطع من المقاييس

In [ ]:
from evaluation.metrics import (
    bootstrap_ci, gwet_ac1, interpret_kappa,
    percent_agreement, weighted_cohens_kappa,
)

def load_pairs(mode='full'):
    pairs = []
    for path in sorted((RESULTS / mode).glob('*.json')):
        run = json.loads(path.read_text(encoding='utf-8'))
        pairs += [tuple(p) for p in (run.get('metrics') or {}).get('matched_pairs', [])]
    return pairs

pairs = load_pairs('full')
if len(pairs) < 2:
    print('⚠️ لا أزواج مطابقة (GT غير مُملأة؟) — املأ evaluation/dataset/ground_truth/*.json')
else:
    a = [p[0] for p in pairs]; b = [p[1] for p in pairs]
    print('n =', len(pairs))
    print('Percent Agreement =', percent_agreement(a, b))
    print('Weighted Kappa    =', weighted_cohens_kappa(a, b),
          '⇒', interpret_kappa(weighted_cohens_kappa(a, b)))
    print("Gwet's AC1        =", gwet_ac1(a, b))
    print('CI95 Kappa =', bootstrap_ci(pairs, lambda s: weighted_cohens_kappa(
        [x for x, _ in s], [y for _, y in s])))

## 3) مصفوفة الالتباس (تحقق متقاطع من الخام)

In [ ]:
if len(pairs) >= 2:
    import pandas as pd
    from evaluation.metrics import confusion_matrix
    labels = ['Low', 'Medium', 'High', 'Critical']
    matrix = confusion_matrix([p[0] for p in pairs], [p[1] for p in pairs], categories=labels)
    pd.DataFrame(matrix, index=[f'النظام: {c}' for c in labels],
                 columns=[f'الخبير: {c}' for c in labels])

## 4) رسوم Plotly (من analyze_results)

In [ ]:
from IPython.display import HTML, display
figures = sorted((RESULTS / 'figures').glob('*.html')) if (RESULTS / 'figures').exists() else []
print('عدد الرسوم:', len(figures))
for figure in figures:
    print('•', figure.name)
if figures:
    display(HTML(figures[0].read_text(encoding='utf-8')))

## 5) منحنى التعلم من جدول التصحيحات

In [ ]:
from evaluation.analyze_feedback import cohort_agreement, load_corrections
try:
    corrections = load_corrections()
    print('التصحيحات البشرية:', len(corrections))
    if len(corrections) >= 2:
        for label, cohort in (('A (بلا أمثلة)', corrections[:20]),
                              ('B (مع أمثلة)', corrections[20:])):
            print(label, '⇒', cohort_agreement(cohort))
except Exception as exc:
    print('⚠️ تعذّرت قراءة التصحيحات:', exc)